# Guided exercise: mini project 1 again, in six lines of pandas

**Week 0, foundations guide, Chapter 6: pandas.** Eight steps that fit the chapter's ninety
minutes of hands-on, build mini project 6, and end on the cells you copy into your
`w00-diagnostic-pandas` repository.

> **Meera Raghavan, CEO of Kalpa Retail:** "Revenue fell from Q1 to Q2, where did it go?"

Chapter 1's exercise answered the first part of her question, revenue by tier, with a loop over ten
order rows. This exercise answers it again from the same ten rows, saved as a CSV and read by
pandas, and the answer has to agree with the loop to the rupee. The chapter's claim is that a
DataFrame is the SQL pipeline with a different spelling: the mask is `WHERE`, `groupby` is
`GROUP BY`, `agg` is the `SELECT` list and `merge` is `JOIN`.

The ten rows are constructed for practice, from amounts the guide itself uses as examples. The two
customer counts in `customers.csv` are the Q1 figures from the diagnostic's case table in Q29, which
is a fictional case, so none of it is Kalpa's data.

**How the placeholders work.** Every `__TODO1__`, `__TODO2__` and so on is yours to replace. A
lettered placeholder lists four options in the comment above it, and you replace it with the code
of the letter you pick. A prediction asks for the value you expect, written before you run the
cell that produces it. The trace table, the change you make on purpose in step 6 and the closing
sentence are yours to write. Run as shipped, the notebook stops with a `NameError` at the first
placeholder, which is where you start, and the checks under each step say whether you were right.

## Step 1. Redraw the DataFrame before any code (8 minutes)

Close the chapter and redraw its picture from memory on paper: a DataFrame with its index down
the side and its columns across the top, where each column is a Series with one dtype, and one
amount that pandas has set to `NaN`. Write the dtype you expect under the amount column. Give it
four minutes before you run the next cell.

The next cell finds the programme's helper, `scripts/c2kit.py`, by walking up from this folder,
and draws the same picture from the first three rows you are about to load, so you can compare
the two drawings column by column.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit

kit.matrix(["0", "1", "2"], ["order_id", "tier", "amount: float64", "status"],
           [["K-101", "Plus", "1200.0", "paid"],
            ["K-102", "Basic", "950.0", "paid"],
            ["K-103", "Plus", "NaN", "paid"]],
           title="A DataFrame: the index down the side, and each column a Series with one dtype")

The index labels each row and survives every filter, which is why the rows a mask keeps still
carry their original numbers. The amount column can be `float64` with a `NaN` in it because `NaN`
is itself a float; if a single value in the column were text, the whole Series would be text,
which is the bug step 3 goes looking for. The map below is Figure 26 of the chapter: the six
moves this exercise runs, each with its SQL twin.

In [ ]:
kit.flow(["read_csv\nfile to DataFrame", "to_numeric\n'n/a' -> NaN", "boolean mask\nWHERE status = 'paid'",
          "groupby\nGROUP BY tier", "agg\nSUM and COUNT", "merge\nJOIN customers"],
         title="The six moves, in the order of the SQL pipeline")

## Step 2. Save the ten rows as two files (6 minutes)

Mini project 6 starts from files, so the first cell writes the ten rows of Chapter 1's exercise
into `orders.csv` and the two customer counts into `customers.csv`. Both files go into an
`output/` folder beside this notebook, which git ignores, so a run leaves the repository as it
was. The cell prints both files after writing them: a CSV is only text, and the quotes around
`"1,200"` are how the file keeps the comma inside an amount from splitting it into two fields.
The cell after it records the three rows planted with amounts no conversion can read, the list
Chapter 1's exercise ended on, for the self-check to count against.

In [ ]:
# Mini project cell one: save the ten rows from mini project 1, and the customers, as CSV files
import csv
import pathlib

import pandas as pd

DATA = pathlib.Path("output")   # in your repository, pathlib.Path(".") keeps the files beside the notebook
DATA.mkdir(exist_ok=True)

rows = [
    ("K-101", "Plus", "1,200", "paid"),
    ("K-102", "Basic", "950", "paid"),
    ("K-103", "Plus", "n/a", "paid"),
    ("K-104", "Basic", "300", "paid"),
    ("K-105", "Plus", "800", "cancelled"),
    ("K-106", "Basic", "900", "paid"),
    ("K-107", "Plus", "", "paid"),
    ("K-108", "Plus", "1,200", "paid"),
    ("K-109", "Basic", "Rs 900", "paid"),
    ("K-110", "Plus", "950", "paid"),
]
customers_rows = [("Plus", 10000), ("Basic", 40000)]

with open(DATA / "orders.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["order_id", "tier", "amount", "status"])
    writer.writerows(rows)
with open(DATA / "customers.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["tier", "customers_q1"])
    writer.writerows(customers_rows)

print((DATA / "orders.csv").read_text())
print((DATA / "customers.csv").read_text())

In [ ]:
planted_bad = ["K-103", "K-107", "K-109"]   # the rows mini project 1 planted with amounts that are not numbers
print("rows planted with bad amounts:", planted_bad)

## Step 3. Load the file and read its types (10 minutes)

`df.dtypes` is the first thing to print after loading anything. Before you run the load, make two
predictions in the next cell: will pandas read the amount column as numbers, and how many amounts
will already be `NaN` the moment the file is read, before any conversion of yours has run?

In [ ]:
# TODO 1. Will pandas read the amount column as numbers? Write True or False.
predicted_numeric = __TODO1__

# TODO 2. How many amounts will already be NaN straight after read_csv? Write a number.
predicted_nan_at_load = __TODO2__
print("predictions recorded:", predicted_numeric, predicted_nan_at_load)

In [ ]:
# Mini project cell two: load the orders, and read the types before anything else
df = pd.read_csv(DATA / "orders.csv")
nan_at_load = int(df["amount"].isna().sum())
print(df.dtypes, end="\n\n")
print("amounts already NaN at load:", nan_at_load)
df

In [ ]:
kit.check("your prediction on the amount column's type matched",
          predicted_numeric == pd.api.types.is_numeric_dtype(df["amount"]), f"dtype {df['amount'].dtype}")
kit.check("your prediction of NaN at load matched", predicted_nan_at_load == nan_at_load,
          f"predicted {predicted_nan_at_load}, read_csv left {nan_at_load}")
kit.check("all ten rows loaded, one per order", len(df) == 10 and df["order_id"].is_unique,
          f"{len(df)} rows")

The dtype prints as `str` in pandas 3 and as `object` in earlier versions, which is what the
guide shows. Either way the column holds text, and every sum on it will be wrong or will fail. If the
`NaN` count surprised you, `read_csv` keeps a list of spellings that mean missing, and an empty
field and the text `n/a` are both on it. `Rs 900` is on no such list, so it arrives as text beside
the good amounts, and step 4's conversion has to deal with it.

## Step 4. Convert the amounts, and count what could not be read (12 minutes)

The chapter's conversion is one line, and it is Chapter 1's `try` per row applied to a whole
column at once: every value that cannot be read becomes `NaN` and stays in the table, where you
can count it. `NaN` behaves like SQL's `NULL` in a total, so `sum()` skips it, and `isna().sum()`
counts the rows that failed, which is the number Anand asked for in Q9. Predict that count in the
next cell before you run the conversion.

In [ ]:
# TODO 3. How many amounts will be NaN after the conversion? Write a number.
predicted_nan_after = __TODO3__
print("prediction recorded:", predicted_nan_after)

In [ ]:
# Mini project cell three: the amounts as numbers, with every value that cannot be read set to NaN
# TODO 4. Which expression turns the amounts into numbers and sets every value it cannot read to NaN?
#   a) pd.to_numeric(df["amount"], errors="coerce")
#   b) pd.to_numeric(df["amount"].str.replace(",", ""))
#   c) pd.to_numeric(df["amount"].str.replace(",", ""), errors="coerce")
#   d) pd.to_numeric(df["amount"].str.replace(",", "").fillna("0"), errors="coerce")
df["amount"] = __TODO4__
nan_after = int(df["amount"].isna().sum())
print("amounts that could not be read, now NaN:", nan_after)
df

In [ ]:
kit.bars([("after read_csv", nan_at_load), ("after to_numeric", nan_after)],
         title="Amounts that are NaN, before and after the conversion", width=560)
kit.check("your prediction of NaN after the conversion matched", predicted_nan_after == nan_after,
          f"predicted {predicted_nan_after}, got {nan_after}")
kit.check("the amount column holds numbers now", pd.api.types.is_float_dtype(df["amount"]),
          f"dtype {df['amount'].dtype}")
kit.check("the NaN rows are exactly the rows planted with bad amounts",
          sorted(df.loc[df["amount"].isna(), "order_id"]) == sorted(planted_bad),
          str(sorted(df.loc[df["amount"].isna(), "order_id"])))
kit.check("no row was dropped: a bad amount becomes NaN and its row stays", len(df) == 10,
          f"{len(df)} rows")

## Step 5. Trace the pipeline by hand, then run it (18 minutes)

The last three moves are the mask, `groupby` with `agg`, and the merge: `WHERE`, `GROUP BY` with
its `SELECT` list, and `JOIN`. Before you run them, fill the trace table by hand. For each move,
write how many rows come out of it and how many `NaN` values remain in the amount column, or in
the revenue column once `agg` has run. The first row is the move you have just run.

In [ ]:
# TODO 5. One tuple per move, in order: (move, rows after it, NaN after it), for the moves
# "mask", "groupby and agg" and "merge". The first row is filled in for you.
my_trace = [
    ("to_numeric", 10, 3),
    __TODO5__
]
print(len(my_trace), "moves traced")

Cell four runs the three moves. The merge takes `validate`, which states the shape the join must
have and raises the moment the tables break it: the first word describes the left table, the paid
orders, and the second the right table, `customers.csv`, where `one` means every key appears once
and `many` allows repeats.

In [ ]:
# Mini project cell four: the paid orders by tier, then the customers joined on tier
# TODO 6. Which expression keeps the paid orders, and only them?
#   a) df[df["status"] == "paid"]
#   b) df[df["amount"].notna()]
#   c) df["status"] == "paid"
#   d) df.dropna()[df.dropna()["status"] == "paid"]
paid = __TODO6__
by_tier = paid.groupby("tier").agg(orders=("order_id", "count"), revenue=("amount", "sum"))

customers = pd.read_csv(DATA / "customers.csv")
# TODO 7. Which validate value makes the merge refuse to run when customers.csv repeats a tier?
#   a) "one_to_one"
#   b) "one_to_many"
#   c) "many_to_many"
#   d) "many_to_one"
check_keys = __TODO7__
merged = paid.merge(customers, on="tier", how="left", validate=check_keys)
print(len(paid), "paid orders went into the merge and", len(merged), "rows came out")
by_tier

In [ ]:
real_trace = [
    ("to_numeric", len(df), int(df["amount"].isna().sum())),
    ("mask", len(paid), int(paid["amount"].isna().sum())),
    ("groupby and agg", len(by_tier), int(by_tier["revenue"].isna().sum())),
    ("merge", len(merged), int(merged["amount"].isna().sum())),
]
kit.table(["move", "rows after it", "NaN after it", "your row matched"],
          [(*real, "yes" if k < len(my_trace) and tuple(my_trace[k]) == real else "no")
           for k, real in enumerate(real_trace)],
          caption="The real trace of the last four moves, against yours")
kit.columns([t[0] for t in real_trace], [("rows after the move", [t[1] for t in real_trace])],
            title="Rows after each of the last four moves")
matched = sum(1 for mine, real in zip(my_trace, real_trace) if tuple(mine) == real)
kit.check("your trace matches the pipeline move for move",
          [tuple(t) for t in my_trace] == real_trace, f"{matched} of {len(real_trace)} moves match")
kit.check("the merge kept one row per paid order", len(merged) == len(paid),
          f"{len(paid)} before, {len(merged)} after")
kit.check("every paid order found its tier in customers.csv", merged["customers_q1"].notna().all())

`orders` counts every paid order in the tier, so Plus shows 5, two of them with amounts nobody
could read. Counting the amount column instead, as `("amount", "count")` would, skips the `NaN`
and gives 3, the number of clean Plus rows mini project 1 kept; step 7 uses that to hold the two
notebooks to the same counts as well as the same rupees.

## Step 6. Break it on purpose: a repeated key fans the rows out (16 minutes)

The customers file arrives again after someone edited it by hand, and the Plus row has been
pasted twice. The next cell builds that file as a DataFrame and runs the validated merge on it,
which refuses, and refusing is the whole job of `validate`.

In [ ]:
customers_twice = pd.concat([customers, customers[customers["tier"] == "Plus"]], ignore_index=True)
print(customers_twice, end="\n\n")
with kit.expect_error() as err:
    paid.merge(customers_twice, on="tier", how="left", validate=check_keys)
kit.check("the validated merge refused the repeated tier", err.name == "MergeError",
          err.message.splitlines()[0] if err.message else "no error: the merge ran")

The tempting change is the one that makes the error go away: delete the `validate` argument and
run the merge again. Make that change on purpose by replacing `__TODO9__` with
`paid.merge(customers_twice, on="tier", how="left")`, and before you run the cell, predict how
many rows the merge will return.

In [ ]:
# TODO 8. Your prediction, written before you run this cell: rows after the merge without validate.
predicted_fanned = __TODO8__

# TODO 9. The tempting change, made on purpose: the same merge with the validate argument deleted.
merged_bad = __TODO9__
revenue_bad = merged_bad.groupby("tier")["amount"].sum()
print(len(paid), "paid orders went into the merge and", len(merged_bad), "rows came out")

**The plausible wrong answer.** The next cell sets revenue by tier from the unvalidated merge
beside revenue from the paid orders themselves.

In [ ]:
kit.table(["tier", "the paid orders", "after the merge without validate"],
          [(t, kit.rupees(by_tier.loc[t, "revenue"]), kit.rupees(revenue_bad[t])) for t in by_tier.index],
          caption="Revenue by tier, before and after the merge without validate")
kit.columns(list(by_tier.index),
            [("the paid orders", [by_tier.loc[t, "revenue"] for t in by_tier.index]),
             ("after the merge without validate", [revenue_bad[t] for t in by_tier.index])],
            fmt=kit.rupees, width=640,
            title="Revenue by tier from the paid orders, and from the unvalidated merge")

**Why it is wrong.** Every Plus order met two Plus rows in the customers file and came out twice,
so the five paid Plus orders became ten rows and Plus revenue doubled, while Basic, which appears
once, stayed where it was. Nothing raised and nothing looks broken. A revenue-by-tier table built
on this merge would tell Meera that Plus brings in about three times what Basic does, when on
these rows the gap is Rs 1,200, and that is the silent duplication the chapter describes, a
doubled revenue column nobody noticed until Finance did. The checks below catch it the way the
chapter says to, with the row count before and after the merge and the revenue it carries, and
`validate` caught it one cell earlier by refusing to run at all.

In [ ]:
kit.check("your prediction of the rows after the merge matched", predicted_fanned == len(merged_bad),
          f"predicted {predicted_fanned}, the merge returned {len(merged_bad)}")
kit.check("caught: the merge returned more rows than there are paid orders", len(merged_bad) > len(paid),
          f"{len(paid)} paid orders, {len(merged_bad)} rows after the merge")
kit.check("caught: revenue after the merge is more than the paid orders hold",
          revenue_bad.sum() > paid["amount"].sum(),
          f"{kit.rupees(revenue_bad.sum())} against {kit.rupees(paid['amount'].sum())}")

**The fix.** Give the customers file back one row per tier before the join, then run the
validated merge again. De-duplicating or aggregating the right-hand table first is the fix the
chapter's interview answer names.

In [ ]:
# TODO 10. Which expression gives customers_twice back one row per tier?
#   a) customers_twice.dropna()
#   b) customers_twice.drop_duplicates(subset="tier")
#   c) customers_twice[customers_twice["tier"] != "Plus"]
#   d) customers_twice.sort_values("tier")
customers_fixed = __TODO10__
merged_fixed = paid.merge(customers_fixed, on="tier", how="left", validate=check_keys)
print(len(paid), "paid orders went into the fixed merge and", len(merged_fixed), "rows came out")
merged_fixed.groupby("tier")["amount"].sum()

In [ ]:
kit.check("the fixed merge keeps one row per paid order", len(merged_fixed) == len(paid),
          f"{len(paid)} before, {len(merged_fixed)} after")
kit.check("every paid order found its tier in the fixed file", merged_fixed["customers_q1"].notna().all())
kit.check("revenue by tier after the fixed merge equals revenue before it",
          (merged_fixed.groupby("tier")["amount"].sum() == by_tier["revenue"]).all())

What changed: the merge is back to nine rows and Plus to Rs 3,350, and a repeated key can no
longer reach a total, because `validate` refuses it before any row is multiplied.

## Step 7. The two assert cells, and the self-check against mini project 1 (12 minutes)

The mini project ends on two assert cells, the chapter's own pair: the group totals add up to the
raw total, and the merge did not fan out. The guide's self-check then asks for three things, each
a check below: the `NaN` count equals the number of bad amounts planted, the tier totals equal
mini project 1's totals, and the notebook runs top to bottom after a restart. That last one holds
only for a run from a fresh kernel, so restart the kernel and run every cell from the top before
you trust it, and delete any cell you added, since the check counts cells.

In [ ]:
# Mini project cell five: the group totals add up to the raw total
assert abs(by_tier["revenue"].sum() - paid["amount"].sum()) < 1e-6
print("revenue by tier sums to", by_tier["revenue"].sum(), "and the paid amounts sum to", paid["amount"].sum())

In [ ]:
# Mini project cell six: the merge did not fan out
assert len(merged) == len(paid)
print(len(paid), "paid orders went into the merge and", len(merged), "rows came out")

To hold this notebook to mini project 1, the next cell runs Chapter 1's cell two on the same ten
rows, the loop the chapter says pandas replaces, and sets its totals beside the pandas ones.

In [ ]:
# Mini project 1's cell two, as Chapter 1's exercise built it, run on the same ten rows
def to_amount(text):
    return float(text.replace(",", ""))


def clean_and_total(rows):
    totals, clean, rejected = {}, [], []
    for row in rows:
        order_id, tier, amount_text, status = row
        try:
            amount = to_amount(amount_text)
        except ValueError:
            rejected.append(row); continue
        if status == "paid":
            totals[tier] = totals.get(tier, 0) + amount
            clean.append((order_id, tier, amount))
    return totals, clean, rejected


loop_totals, loop_clean, loop_rejected = clean_and_total(rows)
kit.table(["tier", "mini project 1's loop", "six lines of pandas"],
          [(t, kit.rupees(loop_totals[t]), kit.rupees(by_tier.loc[t, "revenue"])) for t in loop_totals],
          caption="Revenue by tier for paid orders, computed two ways from the same ten rows")
kit.columns(list(loop_totals),
            [("mini project 1's loop", [loop_totals[t] for t in loop_totals]),
             ("six lines of pandas", [by_tier.loc[t, "revenue"] for t in loop_totals])],
            fmt=kit.rupees, width=560, title="The same ten rows, the same totals")

In [ ]:
cells_run = len(In) - 1   # In holds every cell this kernel has run, after an empty first entry
kit.check("the notebook ran top to bottom once from a fresh kernel, with no error on the way",
          cells_run == 23,
          f"this kernel has run {cells_run} cells, and a clean run reaches this one as cell 23")
kit.check("the NaN count equals the number of bad amounts planted", nan_after == len(planted_bad),
          f"{nan_after} NaN, {len(planted_bad)} planted")
kit.check("the tier totals equal mini project 1's totals, to the rupee",
          set(by_tier.index) == set(loop_totals)
          and all(by_tier.loc[t, "revenue"] == loop_totals[t] for t in loop_totals),
          str({t: float(by_tier.loc[t, "revenue"]) for t in by_tier.index}))
kit.check("both are the totals Chapter 1's exercise printed: Plus Rs 3,350 and Basic Rs 2,150",
          loop_totals == {"Plus": 3350.0, "Basic": 2150.0}, str(loop_totals))
kit.check("the NaN rows are the rows mini project 1 rejected",
          sorted(df.loc[df["amount"].isna(), "order_id"]) == sorted(r[0] for r in loop_rejected))
kit.check("readable amounts per tier match mini project 1's clean rows per tier",
          paid.groupby("tier")["amount"].count().to_dict()
          == {t: sum(1 for c in loop_clean if c[1] == t) for t in loop_totals})

## Step 8. The sentence for Meera, and the copy (8 minutes)

Close the thread where it opened, with the sentence you would now say to Meera, and build it from
the variables so the numbers cannot drift: name both tier totals with `kit.rupees` and say how
many amounts are `NaN`.

In [ ]:
# TODO 11. Your sentence for Meera, as an f-string built from by_tier and nan_after.
say_back = __TODO11__
print(say_back)

In [ ]:
kit.check("your sentence carries both tier totals and the NaN count",
          kit.rupees(by_tier.loc["Plus", "revenue"]) in say_back
          and kit.rupees(by_tier.loc["Basic", "revenue"]) in say_back
          and f" {nan_after} " in f" {say_back} ")

In [ ]:
kit.check_summary()

## Copy it into your repository

Create `w00-diagnostic-pandas` the way Chapter 8 shows, add a notebook named `notebook.ipynb`, and
copy the six cells whose first line starts `# Mini project cell`, from cell one to cell six, in
order, deleting the lettered comment lines as you go. They use nothing from the helper, so they
run anywhere. In the repository, change `DATA = pathlib.Path("output")` to
`DATA = pathlib.Path(".")`, so `orders.csv` and `customers.csv` sit beside the notebook, and
commit both files with it. Restart the kernel and run all: the notebook has to run top to bottom,
report three `NaN` and end on the two asserts.

In Chapter 7's forty-five minutes of hands-on you paste the same rows into a spreadsheet, build
the pivot for revenue by tier, and put a screenshot of it in this repository's README beside the
pandas result. Post the repository link in the Week 0 thread on Discussions.